In [ ]:
from pathlib import Path

# Change these two globals, then Run All.
LOG = Path("logs/20260906T065958Z-ikea-manual-webmcp-102-a9f9f7a6")
SAMPLE_ID = "Shelf/laiva"

CACHE_DIR = None  # Optional Hugging Face dataset cache directory.
PART_ID = None  # None selects the worst officially matched part; otherwise e.g. "00".
CANDIDATE_SEED = None  # None selects the best-PA diagnostic candidate; e.g. "pca-9".
SIMILARITY_POLICY = "geometry"
SIMILARITY_THRESHOLD = 1e-4  # Raw CD before x1000; geometry only.
SIMILARITY_PAIR = None  # Optional pair of IDs, e.g. ("00", "01").
COMPARE_RUN = True  # Compare every saved sample without writing results.

# Inspect assembly metrics

Run with the project environment: `uv sync --extra episodes --group inspection`, then select
`.venv/bin/python` as the notebook kernel in VS Code/Jupyter.

This notebook is read-only with respect to the experiment. It imports the production evaluator,
reconstructs its inputs, reruns its alignment, and exposes the matching and error calculations.
No call to `evaluate_run` is made and no metric files are overwritten.

**Current protocol:** one global SE(3) transform selected by minimum SCD, followed by configurable-group
Hungarian matching. **Diagnostic alternatives:** different global candidates or unrestricted
matching; these are not replacement scores or evidence that all parts are interchangeable.

Outputs contain reconstructed GT. Clear all outputs before saving/sharing this notebook.


In [ ]:
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/assembly_world_agent").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start this notebook inside the assembly-world-agent project.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree

from assembly_world_agent.artifacts import sample_name
from assembly_world_agent.episodes import _render_mesh
from assembly_world_agent.evaluation.geometry import (
    THRESHOLD,
    align,
    chamfer,
    score_parts,
    transform,
)
from assembly_world_agent.evaluation.inspection import compare_run
from assembly_world_agent.evaluation.runner import PROTOCOL, prepare_evaluation_inputs
from assembly_world_agent.loading import load_samples
from assembly_world_agent.similarity import (
    SimilarityConfig,
    inspect_similarity_pair,
    resolve_equivalence,
)
from assembly_world_agent.utils import apply_pose

pd.set_option("display.max_colwidth", None)

LOG = LOG.resolve() if LOG.is_absolute() else (ROOT / LOG).resolve()
pio.renderers.default = "plotly_mimetype+notebook"  # Inline JS; no CDN required.


protected = [
    LOG / "metrics.jsonl",
    LOG / "metrics_summary.json",
    LOG / "metrics.json",
    *LOG.glob("samples/*/final.episode.zip"),
]
input_hashes = {p: hashlib.sha256(p.read_bytes()).hexdigest() for p in protected if p.exists()}
meta = json.loads((LOG / "meta.json").read_text())
identity = meta["config"]["identity"]
expected = meta["config"]["samples"][SAMPLE_ID]
saved_rows = (
    [json.loads(line) for line in (LOG / "metrics.jsonl").read_text().splitlines()]
    if (LOG / "metrics.jsonl").exists()
    else []
)
saved = next((r for r in saved_rows if r["sample_id"] == SAMPLE_ID), None)
print("Run:", LOG, "\nSample:", SAMPLE_ID)
display(PROTOCOL)
similarity_config = SimilarityConfig(SIMILARITY_POLICY, SIMILARITY_THRESHOLD)
display(similarity_config.protocol())

## 1. Locate the qualitative/metric disagreement

Low SCD can coexist with low PA: the shape-level metric ignores part identities, whereas PA
compares assigned parts. SR is strict: one failing part is enough to make SR zero.


In [ ]:
if saved_rows:
    overview = pd.DataFrame(
        [{k: r.get(k) for k in ("sample_id", "status", "SCD", "PA", "SR")} for r in saved_rows]
    )
    display(overview[overview.sample_id == SAMPLE_ID])
    display(overview.query("PA < 0.6").sort_values("SCD").head(12))
    px.scatter(
        overview,
        x="SCD",
        y="PA",
        hover_name="sample_id",
        color="SR",
        title="Saved scores: inspect low-SCD / low-PA samples",
    ).show()

## 2. Reconstruct exactly the production inputs

The shared helper validates episode identity, payload hashes, object catalog and geometry,
restores the final logical state, samples each part, and applies the common scale conversion.
`prediction` and `target` below are already in evaluation units, before global alignment.
The source groups are shown literally: a composite self-group is not an exchangeable part group.


In [ ]:
source = next(
    load_samples(
        identity["dataset"],
        revision=identity["revision"],
        sample_ids=[SAMPLE_ID],
        streaming=False,
        cache_dir=CACHE_DIR,
    )
)
ctx = prepare_evaluation_inputs(
    LOG / "samples" / sample_name(SAMPLE_ID),
    expected,
    identity,
    source,
    similarity=similarity_config,
)
parts, prediction, target = ctx["parts"], ctx["prediction"], ctx["target"]
part_ids = [p.part_id for p in parts]
groups = ctx["groups"]
N = len(parts)
raw_relation = ctx["sample"].annotations.get("geometric_equivalence_relation", {})
if isinstance(raw_relation, str):
    raw_relation = json.loads(raw_relation)
display(raw_relation)
display(
    pd.DataFrame(
        [
            {
                "part_id": p.part_id,
                "annotation_id": p.metadata.get("annotation_part_id"),
                "allowed_targets": [part_ids[j] for g in groups if i in g for j in g],
                "points": len(p.points),
            }
            for i, p in enumerate(parts)
        ]
    )
)
print("Final state:", ctx["state_index"], "| Common scale divisor:", ctx["divisor"])
print("Composite self-relations ignored:", ctx["ignored"])

## Input-shape similarity: groups and threshold edges

Pairwise SE(3) fitting here is used only to discover interchangeable input shapes. Its transforms
never modify assembly scoring. The threshold uses raw CD and the furniture's shared scale.


In [ ]:
source_equivalence = resolve_equivalence(ctx["sample"], config=SimilarityConfig("source"))
geometry_equivalence = (
    ctx["equivalence"]
    if SIMILARITY_POLICY == "geometry"
    else resolve_equivalence(
        ctx["sample"], config=SimilarityConfig("geometry", SIMILARITY_THRESHOLD)
    )
)
display(
    pd.DataFrame(
        [
            dict(
                policy=r["policy"],
                groups=r["groups"],
                source_missing=r["source_annotation_missing"],
            )
            for r in [source_equivalence, geometry_equivalence]
        ]
    )
)
display(pd.DataFrame(geometry_equivalence["group_diagnostics"]))
distance_matrix = np.array(geometry_equivalence["distances"])
fig = go.Figure(
    go.Heatmap(
        z=np.log10(np.maximum(distance_matrix, 1e-12)),
        x=part_ids,
        y=part_ids,
        customdata=distance_matrix,
        colorbar=dict(title="log10 pair CD"),
        hovertemplate="%{y} / %{x}<br>CD=%{customdata:.8f}<extra></extra>",
    )
)
ei, ej = np.where((distance_matrix <= SIMILARITY_THRESHOLD) & ~np.eye(N, dtype=bool))
fig.add_trace(
    go.Scatter(
        x=[part_ids[j] for j in ej],
        y=[part_ids[i] for i in ei],
        mode="markers",
        marker=dict(symbol="circle-open", color="lime", size=12),
        name="Threshold edge",
    )
)
fig.update_layout(title="Input-shape edges; groups are connected components", height=600)
fig.show()
display(pd.DataFrame(geometry_equivalence["registrations"]))

## Inspect a shape pair

Set `SIMILARITY_PAIR` to inspect specific IDs; the default selects the farthest within-group pair.
Both panels have identical axes and preserve relative size. Group diagnostics above expose pairs
merged transitively despite exceeding the threshold.


In [ ]:
worst_groups = [g for g in geometry_equivalence["group_diagnostics"] if g["worst_pair"]]
pair_ids = SIMILARITY_PAIR or (
    max(worst_groups, key=lambda g: g["max_chamfer"])["worst_pair"]
    if worst_groups
    else (part_ids[0], part_ids[min(1, N - 1)])
)
pair = inspect_similarity_pair(ctx["sample"], *pair_ids)
display(pair["alignment"])
print(
    "Pair:",
    pair["part_ids"],
    "CD:",
    pair["alignment"]["chamfer"],
    "SCD:",
    1000 * pair["alignment"]["chamfer"],
    "threshold:",
    SIMILARITY_THRESHOLD,
)
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}] * 2],
    subplot_titles=["Canonical inputs: relative size retained", "After SE(3) registration"],
)
for col, pred in [(1, pair["prediction"]), (2, pair["aligned"])]:
    for name, cloud, color in [
        (pair["part_ids"][0], pred, "#2589e8"),
        (pair["part_ids"][1], pair["target"], "#ed9935"),
    ]:
        fig.add_trace(
            go.Scatter3d(
                x=cloud[:, 0],
                y=cloud[:, 1],
                z=cloud[:, 2],
                mode="markers",
                name=name,
                marker=dict(size=2, color=color),
                showlegend=col == 1,
            ),
            row=1,
            col=col,
        )
all_pair_points = np.concatenate([pair["prediction"], pair["aligned"], pair["target"]])
lo, hi = all_pair_points.min(0), all_pair_points.max(0)
fig.update_scenes(
    aspectmode="data",
    **{a + "axis": dict(range=[lo[k] - 0.01, hi[k] + 0.01]) for k, a in enumerate("xyz")},
)
fig.update_layout(height=520, title="Shape-only inspection; no scaling or reflection").show()

## 3. Inspect the actual meshes before and after alignment

Each panel uses the same axes, equal spatial aspect, and a shared range. Drag to rotate,
scroll to zoom, and use the legend to isolate parts. Prediction/GT panels use source-ID colors;
the overlay uses blue prediction and orange GT. Meshes are for display only: scores use all
1000 sampled points per part, never a simplified display mesh.


In [ ]:
palette = px.colors.qualitative.Dark24
colors = [palette[i % len(palette)] for i in range(N)]
triangles, pred_vertices, gt_vertices = [], [], []
for p, (r, t) in zip(parts, ctx["poses"]):
    vertices, faces = _render_mesh(p)
    triangles.append(faces)
    pred_vertices.append(transform(vertices, r, t) / ctx["divisor"])
    gt_vertices.append(apply_pose(vertices, p.gt_pose) / ctx["divisor"])


def assembly_view(pred, gt, title, pred_colors=None):
    fig = make_subplots(
        rows=1,
        cols=3,
        specs=[[{"type": "scene"}] * 3],
        subplot_titles=["Prediction", "Ground truth", "Overlay"],
    )

    def add(vertices, faces, color, name, col, opacity=1, legend=False):
        fig.add_trace(
            go.Mesh3d(
                x=vertices[:, 0],
                y=vertices[:, 1],
                z=vertices[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color=color,
                opacity=opacity,
                name=name,
                legendgroup=name,
                showlegend=legend,
                hovertemplate=name + "<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}<extra></extra>",
            ),
            row=1,
            col=col,
        )

    for i, pid in enumerate(part_ids):
        add(pred[i], triangles[i], (pred_colors or colors)[i], pid, 1, legend=True)
        add(gt[i], triangles[i], colors[i], pid, 2)
        add(pred[i], triangles[i], "#2589e8", "Prediction " + pid, 3, 0.55)
        add(gt[i], triangles[i], "#ed9935", "GT " + pid, 3, 0.4)
    all_vertices = np.concatenate([*pred, *gt])
    low, high = all_vertices.min(0), all_vertices.max(0)
    pad = max(float(np.max(high - low)) * 0.06, 0.01)
    axes = {
        axis + "axis": dict(title=axis, range=[low[k] - pad, high[k] + pad])
        for k, axis in enumerate("xyz")
    }
    fig.update_scenes(
        **axes,
        aspectmode="data",
        camera=dict(up=dict(x=0, y=0, z=1), eye=dict(x=1.4, y=-1.8, z=1.2)),
    )
    fig.update_layout(height=540, title=title, margin=dict(l=0, r=0, b=0, t=65))
    return fig


assembly_view(pred_vertices, gt_vertices, "Before global alignment — evaluation units").show()

## 4. Run the production global alignment

The extra flag only returns candidate transforms for inspection; it does not change the
optimization or selected candidate. No per-part transforms are fitted during scoring.


In [ ]:
rotation, translation, alignment = align(
    np.concatenate(prediction),
    np.concatenate(target),
    ctx["seeds"],
    include_candidate_transforms=True,
)
aligned = [transform(p, rotation, translation) for p in prediction]
aligned_vertices = [transform(v, rotation, translation) for v in pred_vertices]
official, part_records = score_parts(aligned, target, groups, part_ids, alignment["chamfer"])
print(
    "Selected seed:",
    alignment["seed"],
    "| Iterations:",
    alignment["iterations"],
    "| Converged:",
    alignment["converged"],
)
display(pd.DataFrame(rotation, columns=["x", "y", "z"]))
print("Translation:", translation)
display(pd.DataFrame([official], index=["Recomputed current protocol"]))
if (
    saved
    and saved.get("status") == "scored"
    and saved.get("protocol_version") == PROTOCOL["version"]
    and saved.get("similarity", {}).get("protocol") == similarity_config.protocol()
):
    assert saved["episode_sha256"] == ctx["checksum"], "Saved scores refer to another episode."
    assert saved["protocol_version"] == PROTOCOL["version"], "Saved protocol differs."
    np.testing.assert_allclose(
        [official[k] for k in ("SCD", "PA", "SR")],
        [saved[k] for k in ("SCD", "PA", "SR")],
        rtol=0,
        atol=1e-10,
    )
    assert [(r["part_id"], r["target_part_id"]) for r in part_records] == [
        (r["part_id"], r["target_part_id"]) for r in saved["parts"]
    ]
    print("Recomputed metrics and matching agree with the saved result.")
else:
    print("Saved configuration differs or no score exists; showing this inspection configuration.")
assembly_view(aligned_vertices, gt_vertices, "Official SCD-selected global alignment").show()

In [ ]:
policy_scores, policy_matches = [], []
for result in [source_equivalence, geometry_equivalence]:
    indices = [[part_ids.index(pid) for pid in g] for g in result["groups"]]
    values, matches = score_parts(aligned, target, indices, part_ids, alignment["chamfer"])
    policy_scores.append(dict(policy=result["policy"], **values))
    policy_matches.extend(dict(policy=result["policy"], **r) for r in matches)
display(pd.DataFrame(policy_scores))
display(
    pd.DataFrame(policy_matches).pivot(
        index="part_id", columns="policy", values=["target_part_id", "chamfer", "correct"]
    )
)

## 5. SCD: where do the point distances go?

For complete shapes P and G, `SCD = 1000 * (mean(min ||p-g||²) + mean(min ||g-p||²))`.
Both directions are averaged separately and added. The plot uses squared distances; the PA
threshold applies to a whole part's two directional means, not to individual point distances.


In [ ]:
P, G = np.concatenate(aligned), np.concatenate(target)
p_to_g = cKDTree(G).query(P)[0] ** 2
g_to_p = cKDTree(P).query(G)[0] ** 2
scd_components = pd.DataFrame(
    [
        {"direction": "prediction -> GT", "mean_squared_distance": p_to_g.mean()},
        {"direction": "GT -> prediction", "mean_squared_distance": g_to_p.mean()},
    ]
)
display(scd_components)
np.testing.assert_allclose(1000 * (p_to_g.mean() + g_to_p.mean()), official["SCD"], atol=1e-10)
fig = go.Figure()
for label, values in [("prediction -> GT", p_to_g), ("GT -> prediction", g_to_p)]:
    fig.add_trace(
        go.Histogram(x=values, name=label, nbinsx=70, opacity=0.55, histnorm="probability")
    )
fig.update_layout(
    title="Whole-shape nearest-neighbor squared distances",
    barmode="overlay",
    xaxis_title="Squared distance",
    yaxis_title="Probability",
).show()

## 6. Hungarian cost matrix and allowed assignments

Row i is a predicted part; column j is a GT part. Every cell is its **full bidirectional
squared Chamfer cost**, with no clipping. Official matching only searches within the selected
same-group blocks. Gray crosses mark forbidden assignments; green circles mark official choices.
The display uses log10(cost) to reveal small and large values; hover shows the raw cost.


In [ ]:
cost = np.array([[chamfer(p, g) for g in target] for p in aligned])
allowed = np.zeros((N, N), dtype=bool)
for group in groups:
    allowed[np.ix_(group, group)] = True
assignment = np.array([part_ids.index(r["target_part_id"]) for r in part_records])
fig = go.Figure(
    go.Heatmap(
        z=np.log10(np.maximum(cost, 1e-12)),
        x=part_ids,
        y=part_ids,
        customdata=cost,
        colorbar=dict(title="log10 CD"),
        hovertemplate="pred=%{y} -> GT=%{x}<br>CD=%{customdata:.7f}<extra></extra>",
    )
)
fi, fj = np.where(~allowed)
fig.add_trace(
    go.Scatter(
        x=[part_ids[j] for j in fj],
        y=[part_ids[i] for i in fi],
        mode="markers",
        marker=dict(symbol="x", size=5, color="gray"),
        name="Forbidden by selected groups",
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=[part_ids[j] for j in assignment],
        y=part_ids,
        mode="markers",
        marker=dict(symbol="circle-open", size=15, color="lime", line=dict(width=2)),
        name="Official Hungarian assignment",
    )
)
fig.update_layout(
    title="Official assignment constraints",
    xaxis_title="GT part",
    yaxis_title="Predicted part",
    height=650,
)
fig.update_yaxes(autorange="reversed")
fig.show()
part_table = pd.DataFrame(part_records)
part_table["identity_CD"] = np.diag(cost)
part_table["best_any_target"] = [part_ids[j] for j in cost.argmin(1)]
part_table["best_any_CD"] = cost.min(1)
part_table["best_any_allowed"] = allowed[np.arange(N), cost.argmin(1)]
part_table["threshold_ratio"] = part_table.chamfer / THRESHOLD
display(part_table.sort_values("chamfer", ascending=False))
assembly_view(
    aligned_vertices,
    gt_vertices,
    "Hungarian matching: prediction colors follow assigned GT IDs",
    pred_colors=[colors[j] for j in assignment],
).show()

## 7. PA and SR, and a diagnostic relaxation

PA is the fraction of officially matched part CDs <= 0.01. SR is 1 only when every part passes.
Unrestricted Hungarian below is a **counterfactual diagnostic**: it can assign unrelated shapes
and is not valid evidence of interchangeability. A large improvement suggests inspecting missing
equivalence groups, the similarity threshold, and/or a symmetry-related global alignment before changing any protocol.


In [ ]:
passes = part_table.chamfer.to_numpy() <= THRESHOLD
print(f"PA = {passes.sum()}/{N} = {passes.mean():.6f}; SR = all(pass) = {int(passes.all())}")
assert official["PA"] == passes.mean() and official["SR"] == int(passes.all())
fig = px.bar(
    part_table,
    x="part_id",
    y="chamfer",
    color="correct",
    hover_data=["target_part_id", "threshold_ratio"],
    title="Official matched part errors",
)
fig.add_hline(y=THRESHOLD, line_dash="dash", annotation_text="PA threshold = 0.01")
fig.update_yaxes(type="log")
fig.show()
relaxed, relaxed_records = score_parts(
    aligned, target, [list(range(N))], part_ids, alignment["chamfer"]
)
display(
    pd.DataFrame([official, relaxed], index=["Selected groups", "Diagnostic unrestricted matching"])
)
relaxed_table = pd.DataFrame(relaxed_records)
relaxed_table["allowed_officially"] = [
    allowed[i, part_ids.index(r["target_part_id"])] for i, r in enumerate(relaxed_records)
]
display(relaxed_table)

## 8. Inspect one failing part and its possible targets

Edit `PART_ID` at the top, or let this cell select the worst official match. Both panels keep the
prediction fixed: left shows its official target, right the unrestricted one-to-one target.
No pairwise realignment is applied. The first panel also shows prediction-to-target nearest-neighbor
segments for a deterministic subset; scoring always uses all points and both directions.


In [ ]:
pid = PART_ID or part_table.sort_values("chamfer", ascending=False).iloc[0].part_id
i = part_ids.index(pid)
j = assignment[i]
k = part_ids.index(relaxed_records[i]["target_part_id"])
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}] * 2],
    subplot_titles=[
        f"Official: {pid} -> {part_ids[j]}",
        f"Diagnostic unrestricted: {pid} -> {part_ids[k]}",
    ],
)
for col, match in [(1, j), (2, k)]:
    for name, cloud, color in [
        ("Prediction", aligned[i], "#2589e8"),
        ("GT", target[match], "#ed9935"),
    ]:
        fig.add_trace(
            go.Scatter3d(
                x=cloud[:, 0],
                y=cloud[:, 1],
                z=cloud[:, 2],
                mode="markers",
                marker=dict(size=2, color=color),
                name=name,
                showlegend=col == 1,
            ),
            row=1,
            col=col,
        )
nearest = cKDTree(target[j]).query(aligned[i])[1]
segments = np.array(
    [v for a, b in zip(aligned[i][::25], target[j][nearest][::25]) for v in (a, b, [np.nan] * 3)]
)
fig.add_trace(
    go.Scatter3d(
        x=segments[:, 0],
        y=segments[:, 1],
        z=segments[:, 2],
        mode="lines",
        line=dict(color="gray", width=2),
        name="Nearest neighbors",
    ),
    row=1,
    col=1,
)
vertices = np.concatenate([aligned[i], target[j], target[k]])
low, high = vertices.min(0), vertices.max(0)
pad = max(float(np.max(high - low)) * 0.08, 0.01)
fig.update_scenes(
    **{a + "axis": dict(range=[low[q] - pad, high[q] + pad]) for q, a in enumerate("xyz")},
    aspectmode="data",
    camera=dict(up=dict(x=0, y=0, z=1), eye=dict(x=1.4, y=-1.8, z=1.2)),
)
fig.update_layout(
    height=570, title="Part pair inspection — shared coordinates, no per-part fitting"
).show()

## 9. Does the SCD objective select a different global symmetry?

Score every already-computed global alignment candidate with the **same selected groups**.
A different candidate may have slightly larger SCD but much larger PA. This is an objective-selection
issue, not a Hungarian implementation error. The best-PA candidate is shown only for diagnosis;
the official candidate is still chosen exclusively by minimum SCD.


In [ ]:
candidate_rows = []
for candidate in alignment["candidates"]:
    r, t = np.array(candidate["rotation"]), np.array(candidate["translation"])
    alternative = [transform(p, r, t) for p in prediction]
    values, _ = score_parts(alternative, target, groups, part_ids, candidate["chamfer"])
    candidate_rows.append(
        dict(
            seed=candidate["seed"],
            **values,
            selected=candidate["seed"] == alignment["seed"],
            iterations=candidate["iterations"],
            converged=candidate["converged"],
        )
    )
candidate_table = pd.DataFrame(candidate_rows)
display(candidate_table.sort_values(["SCD", "seed"]).head(12))
display(candidate_table.sort_values(["PA", "SCD"], ascending=[False, True]).head(12))
px.scatter(
    candidate_table,
    x="SCD",
    y="PA",
    color="selected",
    hover_name="seed",
    hover_data=["SR", "iterations", "converged"],
    title="One global transform per candidate",
).show()
chosen_seed = (
    CANDIDATE_SEED
    or candidate_table.sort_values(["PA", "SCD"], ascending=[False, True]).iloc[0].seed
)
chosen = next(c for c in alignment["candidates"] if c["seed"] == chosen_seed)
r, t = np.array(chosen["rotation"]), np.array(chosen["translation"])
assembly_view(
    [transform(v, r, t) for v in pred_vertices],
    gt_vertices,
    f"Diagnostic candidate {chosen_seed}; official choice remains {alignment['seed']}",
).show()

## Full-run comparison at saved geometry alignments

Run geometry evaluation first. This cell validates each episode and recomputes source/geometry
assignments at the recorded global transform. It does not rerun grouping or write any files.
Errors remain visible; averages use successfully compared samples.


In [ ]:
if COMPARE_RUN:
    print(
        "Full-run comparison uses the saved geometry configuration:",
        saved_rows[0].get("similarity", {}).get("protocol") if saved_rows else None,
    )
    comparison = pd.DataFrame(compare_run(LOG, cache_dir=CACHE_DIR))
    with pd.option_context("display.max_rows", None):
        display(comparison)
    good = comparison[comparison.status == "scored"]
    print(
        "Expected:",
        len(meta["config"]["samples"]),
        "compared:",
        len(good),
        "errors:",
        len(comparison) - len(good),
    )
    if len(good):
        display(
            good[["SCD", "source_PA", "geometry_PA", "source_SR", "geometry_SR"]]
            .mean()
            .to_frame("sample mean")
        )
        display(good.sort_values("delta_PA", ascending=False).head(20))
        display(good[good.above_threshold_pair_count > 0])
    display(comparison[comparison.status != "scored"])

## 10. Audit and interpretation

Use the tables to distinguish: (1) genuinely displaced/misoriented geometry; (2) source or geometry groups
that do not allow an intuitively equivalent exchange; (3) a whole-shape symmetry preferred by SCD
but penalized by ID-sensitive PA; and (4) errors close to the fixed threshold.

Do not conclude that a relaxed match is semantically correct from cost alone. Inspect the source
annotation, meshes and candidate pose together. Changes to matching or candidate selection require
an explicitly revised evaluation protocol, not edits to the existing score files.


In [ ]:
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == h for p, h in input_hashes.items())
print("Protected episode and metric files are unchanged.")
display(
    pd.DataFrame(
        [
            dict(mode="Official", **official),
            dict(mode="Diagnostic unrestricted at official alignment", **relaxed),
            dict(
                mode="Diagnostic best selected-group PA among candidates",
                **candidate_table.sort_values(["PA", "SCD"], ascending=[False, True])
                .iloc[0][["SCD", "PA", "SR"]]
                .to_dict(),
            ),
        ]
    )
)
print(
    "Clear all outputs before saving/sharing; this notebook keeps reconstructed GT in memory only."
)